# Retail Data ELT Lab — SQLite and Python

**Student starter notebook**

Use this notebook with the lab instructions to load the instructor-provided CSV files into SQLite, inspect their schemas, create staging views, run data-quality checks, and answer business questions.

**Tools:** Python, pandas, SQLite, Google Colab. No BigQuery or dbt setup is required.

Before starting, put `customers.csv`, `products.csv`, `orders_raw.csv`, and `orders_clean.csv` in a folder in your Google Drive, or use the direct-upload option in Step 1. Run cells from top to bottom. Use actual column names from your files; examples below are templates.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd

print("Python and SQLite are ready.")
print("SQLite version:", sqlite3.sqlite_version)
print("pandas version:", pd.__version__)

Python and SQLite are ready.
SQLite version: 3.45.1
pandas version: 2.2.3


## 1. Make the CSV files available

Choose **one** option: Google Drive (recommended) or direct upload. If using Drive, copy the four CSV files into a folder you own, for example `MyDrive/Retail_ELT_Lab`.

In [ ]:
# OPTION A (recommended): Mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/Retail_ELT_Lab')  # Change if your folder differs

required_files = [
    'customers.csv',
    'products.csv',
    'orders_raw.csv',
    'orders_clean.csv'
]

for filename in required_files:
    file_path = DATA_DIR / filename
    print(f"{filename}: {'FOUND' if file_path.exists() else 'NOT FOUND'}")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!git clone https://github.com/SasankaPandaSCIT/Data-Engineering-course.git /content/course_repo
!find /content/course_repo -type f | head -60

Cloning into '/content/course_repo'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), 6.27 KiB | 3.13 MiB/s, done.
/content/course_repo/.git/index
/content/course_repo/.git/HEAD
/content/course_repo/.git/packed-refs
/content/course_repo/.git/objects/pack/pack-7d5ccf263937676303d1dd686c044d07f474c17f.rev
/content/course_repo/.git/objects/pack/pack-7d5ccf263937676303d1dd686c044d07f474c17f.pack
/content/course_repo/.git/objects/pack/pack-7d5ccf263937676303d1dd686c044d07f474c17f.idx
/content/course_repo/.git/hooks/post-update.sample
/content/course_repo/.git/hooks/commit-msg.sample
/content/course_repo/.git/hooks/fsmonitor-watchman.sample
/content/course_repo/.git/hooks/pre-push.sample
/content/course_repo/.git/hooks/pre-receive.sample
/content/course_repo/.git/hooks/pre-rebase.sample
/content/course_repo/.git

In [ ]:
!pip install -q gdown
!gdown --folder https://drive.google.com/drive/folders/1Z3Ex1mGX4ut6MJ-QVlKQwinEy8oBeo7g -O /content/lab_data
!ls -la /content/lab_data

Retrieving folder contents
Processing file 1SSTA9JGaFOTtcaNQH-JvUNNtNINlVWIU customers.csv
Processing file 1ILjGx801NGNfl1WIPmQ3Nov2kbPRVzs- generate.py
Processing file 1OoRT6OJ6YHJOZ6pvIbF1my4cIEuzZO_X orders_clean.csv
Processing file 1baxxYyHC81mYAqIyF6GzUvm8ArDqY4tC orders_raw.csv
Processing file 1p4aA_wYcwkk7635FhJfkDE2ofeVazWOE products.csv
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1SSTA9JGaFOTtcaNQH-JvUNNtNINlVWIU
To: /content/lab_data/customers.csv
100% 33.7k/33.7k [00:00<00:00, 59.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1ILjGx801NGNfl1WIPmQ3Nov2kbPRVzs-
From (redirected): https://drive.google.com/uc?id=1ILjGx801NGNfl1WIPmQ3Nov2kbPRVzs-&confirm=t&uuid=3f9b684d-dc5b-46d8-b168-332060b32b7c
To: /content/lab_data/generate.py
100% 4.56k/4.56k [00:00<00:00, 16.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1OoRT6OJ6YHJOZ6pvIbF1my4

In [ ]:
# Files were downloaded from the shared Drive folder with gdown because Drive mount failed in this browser.
DATA_DIR = Path('/content/lab_data')
print('customers.csv:', (DATA_DIR / 'customers.csv').exists())
print('products.csv:', (DATA_DIR / 'products.csv').exists())
print('orders_raw.csv:', (DATA_DIR / 'orders_raw.csv').exists())
print('orders_clean.csv:', (DATA_DIR / 'orders_clean.csv').exists())

customers.csv: True
products.csv: True
orders_raw.csv: True
orders_clean.csv: True


In [ ]:
# OPTION B: Upload files directly from your computer.
# Run this cell ONLY if you are not using Google Drive.
# from google.colab import files
# uploaded = files.upload()
# DATA_DIR = Path('/content')
#
# After upload, verify that all four CSV files are in /content.

## 2. Read and inspect the CSV files

Run this after all four files show as found (or after direct upload). Review the columns and sample records before writing SQL.

In [ ]:
customers_df = pd.read_csv(DATA_DIR / 'customers.csv')
products_df = pd.read_csv(DATA_DIR / 'products.csv')
orders_raw_df = pd.read_csv(DATA_DIR / 'orders_raw.csv')
orders_clean_df = pd.read_csv(DATA_DIR / 'orders_clean.csv')

dataframes = {
    'customers': customers_df,
    'products': products_df,
    'orders_raw': orders_raw_df,
    'orders_clean': orders_clean_df
}

for name, df in dataframes.items():
    print(f"\n--- {name} ---")
    print("Shape (rows, columns):", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head())


--- customers ---
Shape (rows, columns): (500, 7)
Columns: ['customer_id', 'first_name', 'last_name', 'email', 'city', 'state', 'signup_date']


,customer_id,first_name,last_name,email,city,state,signup_date
0,C0001,James,Gupta,james.gupta1@example.com,Dallas,TX,2023-07-18
1,C0002,Krishna,Khan,krishna.khan2@example.com,Houston,TX,2022-07-29
2,C0003,Maria,Davis,maria.davis3@example.com,Austin,TX,2024-05-14
3,C0004,Aisha,Sharma,aisha.sharma4@example.com,Austin,TX,2023-03-24
4,C0005,Krishna,Brown,krishna.brown5@example.com,Boston,MA,2022-02-24



--- products ---
Shape (rows, columns): (20, 4)
Columns: ['product_id', 'product_name', 'category', 'price']


,product_id,product_name,category,price
0,P001,Wireless Headphones,Electronics,89.99
1,P002,Bluetooth Speaker,Electronics,49.99
2,P003,Laptop Stand,Electronics,34.99
3,P004,USB-C Hub,Electronics,29.99
4,P005,Smart Watch,Electronics,199.99



--- orders_raw ---
Shape (rows, columns): (5100, 6)
Columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'order_date', 'status']


,order_id,customer_id,product_id,quantity,order_date,status
0,O02814,C0472,P011,4,2025-07-15,pending
1,O04017,C0492,P020,5,2025-04-30,pending
2,O03993,C0065,P001,5,2025-01-10,delivered
3,O04314,C0107,P011,1,2025-07-03,delivered
4,O01732,C0175,P019,2,2024-04-28,shipped



--- orders_clean ---
Shape (rows, columns): (5000, 6)
Columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'order_date', 'status']


,order_id,customer_id,product_id,quantity,order_date,status
0,O00001,C0397,P009,4,2025-09-18,delivered
1,O00002,C0147,P016,2,2024-03-16,delivered
2,O00003,C0232,P006,4,2025-11-29,cancelled
3,O00004,C0496,P011,3,2025-07-17,shipped
4,O00005,C0278,P010,3,2025-12-29,pending


In [ ]:
for name, df in dataframes.items():
    print(f"\n--- {name}: data types ---")
    display(df.dtypes.to_frame('data_type'))
    print("Missing values by column:")
    display(df.isna().sum().to_frame('missing_count'))


--- customers: data types ---


,data_type
customer_id,object
first_name,object
last_name,object
email,object
city,object
state,object
signup_date,object


Missing values by column:


,missing_count
customer_id,0
first_name,0
last_name,0
email,0
city,0
state,0
signup_date,0



--- products: data types ---


,data_type
product_id,object
product_name,object
category,object
price,float64


... (6 outputs left out)


--- orders_clean: data types ---


,data_type
order_id,object
customer_id,object
product_id,object
quantity,int64
order_date,object
status,object


Missing values by column:


,missing_count
order_id,0
customer_id,0
product_id,0
quantity,0
order_date,0
status,0


## 3. Create a SQLite database and load the raw tables

Load `customers.csv`, `products.csv`, and `orders_raw.csv` into SQLite. Keep `orders_clean.csv` as a reference file for later comparison; do not use it to replace the raw orders table.

In [ ]:
DB_PATH = Path('/content/retail_lab.db')
conn = sqlite3.connect(DB_PATH)

raw_dataframes = {
    'customers': customers_df,
    'products': products_df,
    'orders_raw': orders_raw_df
}

for table_name, df in raw_dataframes.items():
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Loaded {len(df):,} rows into {table_name}")

print("SQLite database:", DB_PATH)

Loaded 500 rows into customers
Loaded 20 rows into products
Loaded 5,100 rows into orders_raw
SQLite database: /content/retail_lab.db


In [ ]:
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", conn)
display(tables)

for table_name in ['customers', 'products', 'orders_raw']:
    count = pd.read_sql_query(
        f'SELECT COUNT(*) AS row_count FROM "{table_name}"',
        conn
    ).iloc[0]['row_count']
    print(f"{table_name}: {count:,} rows")

,name
0,customers
1,orders_raw
2,products


customers: 500 rows
products: 20 rows
orders_raw: 5,100 rows


In [ ]:
display(pd.read_sql_query('SELECT * FROM orders_raw LIMIT 10;', conn))

,order_id,customer_id,product_id,quantity,order_date,status
0,O02814,C0472,P011,4,2025-07-15,pending
1,O04017,C0492,P020,5,2025-04-30,pending
2,O03993,C0065,P001,5,2025-01-10,delivered
3,O04314,C0107,P011,1,2025-07-03,delivered
4,O01732,C0175,P019,2,2024-04-28,shipped
5,O03546,C0343,P007,5,2024-01-27,delivered
6,O02403,C0189,P020,4,2025-07-28,pending
7,O04813,C0455,P015,3,2024-11-04,shipped
8,O01179,C0424,P010,3,2025-12-05,delivered
9,O00202,C0103,P019,5,2025-07-17,delivered


## 4. Inspect SQLite schemas and table grain

Review the exact field names and data types. Decide what one row represents in each table (for example, one customer, one product, one order, or one order line).

In [ ]:
for table_name in ['customers', 'products', 'orders_raw']:
    print(f"\n--- {table_name} schema ---")
    display(pd.read_sql_query(f'PRAGMA table_info("{table_name}");', conn))


--- customers schema ---


,cid,name,type,notnull,dflt_value,pk
0,0,customer_id,TEXT,0,None,0
1,1,first_name,TEXT,0,None,0
2,2,last_name,TEXT,0,None,0
3,3,email,TEXT,0,None,0
4,4,city,TEXT,0,None,0
5,5,state,TEXT,0,None,0
6,6,signup_date,TEXT,0,None,0



--- products schema ---


,cid,name,type,notnull,dflt_value,pk
0,0,product_id,TEXT,0,None,0
1,1,product_name,TEXT,0,None,0
2,2,category,TEXT,0,None,0
3,3,price,REAL,0,None,0



--- orders_raw schema ---


,cid,name,type,notnull,dflt_value,pk
0,0,order_id,TEXT,0,None,0
1,1,customer_id,TEXT,0,None,0
2,2,product_id,TEXT,0,None,0
3,3,quantity,INTEGER,0,None,0
4,4,order_date,TEXT,0,None,0
5,5,status,TEXT,0,None,0


In [ ]:
# Exploration before designing staging views: grain, keys, orphans, value ranges.
print('RAW grain: rows, distinct order_id, distinct customer_id, distinct product_id')
display(pd.read_sql_query('SELECT COUNT(*) AS rows_raw, COUNT(DISTINCT order_id) AS distinct_order_ids, COUNT(DISTINCT customer_id) AS distinct_customer_ids, COUNT(DISTINCT product_id) AS distinct_product_ids FROM orders_raw;', conn))
print('RAW order_id values that repeat')
display(pd.read_sql_query('SELECT COUNT(*) AS duplicate_order_ids FROM (SELECT order_id FROM orders_raw GROUP BY order_id HAVING COUNT(*) > 1);', conn))
print('RAW repeated order_id detail (top 10)')
display(pd.read_sql_query('SELECT order_id, COUNT(*) AS rows_per_order FROM orders_raw GROUP BY order_id HAVING COUNT(*) > 1 ORDER BY rows_per_order DESC LIMIT 10;', conn))
print('RAW distinct full rows (vs 5100 rows)')
display(pd.read_sql_query('SELECT COUNT(*) AS distinct_full_rows FROM (SELECT DISTINCT order_id, customer_id, product_id, quantity, order_date, status FROM orders_raw);', conn))
print('RAW duplicate customer_id in customers')
display(pd.read_sql_query('SELECT COUNT(*) AS dup_customers FROM (SELECT customer_id FROM customers GROUP BY customer_id HAVING COUNT(*) > 1);', conn))
print('RAW duplicate product_id in products')
display(pd.read_sql_query('SELECT COUNT(*) AS dup_products FROM (SELECT product_id FROM products GROUP BY product_id HAVING COUNT(*) > 1);', conn))
print('RAW orders with missing or blank customer_id')
display(pd.read_sql_query("SELECT COUNT(*) AS missing_customer_id FROM orders_raw WHERE customer_id IS NULL OR TRIM(customer_id) = '';", conn))
print('RAW orders whose customer_id is not in customers (excluding missing)')
display(pd.read_sql_query('SELECT COUNT(*) AS orphan_customer FROM orders_raw o LEFT JOIN customers c ON o.customer_id = c.customer_id WHERE o.customer_id IS NOT NULL AND c.customer_id IS NULL;', conn))
print('RAW orders whose product_id is not in products')
display(pd.read_sql_query('SELECT COUNT(*) AS orphan_product FROM orders_raw o LEFT JOIN products p ON o.product_id = p.product_id WHERE p.product_id IS NULL;', conn))
print('RAW quantity range and non-positive count')
display(pd.read_sql_query('SELECT MIN(quantity) AS min_qty, MAX(quantity) AS max_qty, SUM(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END) AS non_positive_qty FROM orders_raw;', conn))
print('RAW status values')
display(pd.read_sql_query('SELECT status, COUNT(*) AS n FROM orders_raw GROUP BY status ORDER BY n DESC;', conn))
print('RAW order_date range, missing, non-ISO-length')
display(pd.read_sql_query("SELECT MIN(order_date) AS min_date, MAX(order_date) AS max_date, SUM(CASE WHEN order_date IS NULL OR TRIM(order_date) = '' THEN 1 ELSE 0 END) AS missing_date, SUM(CASE WHEN length(order_date) != 10 THEN 1 ELSE 0 END) AS odd_length_date FROM orders_raw;", conn))
print('RAW product categories')
display(pd.read_sql_query('SELECT category, COUNT(*) AS n FROM products GROUP BY category ORDER BY n DESC;', conn))
print('RAW padded text checks')
display(pd.read_sql_query('SELECT COUNT(*) AS padded_customers FROM customers WHERE first_name != TRIM(first_name) OR last_name != TRIM(last_name) OR city != TRIM(city) OR state != TRIM(state);', conn))
display(pd.read_sql_query("SELECT COUNT(*) AS padded_status FROM orders_raw WHERE status != TRIM(status);", conn))
print('CLEAN key facts (pandas)')
print('clean rows:', len(orders_clean_df))
print('clean distinct order_id:', orders_clean_df['order_id'].nunique())
print('clean duplicate full rows:', int(orders_clean_df.duplicated().sum()))
print('clean missing customer_id:', int(orders_clean_df['customer_id'].isna().sum()))
print('CLEAN status values:')
print(orders_clean_df['status'].value_counts().to_string())

RAW grain: rows, distinct order_id, distinct customer_id, distinct product_id


,rows_raw,distinct_order_ids,distinct_customer_ids,distinct_product_ids
0,5100,5000,500,20


RAW order_id values that repeat


,duplicate_order_ids
0,100


RAW repeated order_id detail (top 10)


,order_id,rows_per_order
0,O04994,2
1,O04974,2
2,O04949,2
3,O04864,2
4,O04771,2
5,O04770,2
6,O04691,2
7,O04673,2
8,O04666,2
9,O04618,2


... (20 outputs left out)

RAW padded text checks


,padded_customers
0,0


,padded_status
0,0


CLEAN key facts (pandas)
clean rows: 5000
clean distinct order_id: 5000
clean duplicate full rows: 0
clean missing customer_id: 0
CLEAN status values:
status
delivered    3462
shipped       791
pending       490
cancelled     257


## 5. Create staging views with SQL

These starter views select all columns so they can run before you know the final field names. Improve them by selecting fields needed for analysis, standardizing text where appropriate, and documenting quality rules. If you edit a view definition, rerun its cell; each cell drops and recreates the view.

In [ ]:
# Improved staging view: trim text, blank string -> NULL.
# Grain: one row per customer; customer_id is the candidate business key (500 rows, no duplicates).
conn.execute("DROP VIEW IF EXISTS stg_customers;")
conn.execute("""
CREATE VIEW stg_customers AS
SELECT
NULLIF(TRIM(customer_id), '') AS customer_id,
NULLIF(TRIM(first_name), '') AS first_name,
NULLIF(TRIM(last_name), '') AS last_name,
NULLIF(TRIM(email), '') AS email,
NULLIF(TRIM(city), '') AS city,
NULLIF(TRIM(state), '') AS state,
NULLIF(TRIM(signup_date), '') AS signup_date
FROM customers;
""")
conn.commit()
display(pd.read_sql_query('SELECT * FROM stg_customers LIMIT 5;', conn))

,customer_id,first_name,last_name,email,city,state,signup_date
0,C0001,James,Gupta,james.gupta1@example.com,Dallas,TX,2023-07-18
1,C0002,Krishna,Khan,krishna.khan2@example.com,Houston,TX,2022-07-29
2,C0003,Maria,Davis,maria.davis3@example.com,Austin,TX,2024-05-14
3,C0004,Aisha,Sharma,aisha.sharma4@example.com,Austin,TX,2023-03-24
4,C0005,Krishna,Brown,krishna.brown5@example.com,Boston,MA,2022-02-24


In [ ]:
# Improved staging view: trim text, blank string -> NULL, price as REAL.
# Grain: one row per product; product_id is the candidate business key (20 rows, no duplicates).
conn.execute("DROP VIEW IF EXISTS stg_products;")
conn.execute("""
CREATE VIEW stg_products AS
SELECT
NULLIF(TRIM(product_id), '') AS product_id,
NULLIF(TRIM(product_name), '') AS product_name,
NULLIF(TRIM(category), '') AS category,
CAST(price AS REAL) AS price
FROM products;
""")
conn.commit()
display(pd.read_sql_query('SELECT * FROM stg_products LIMIT 5;', conn))

,product_id,product_name,category,price
0,P001,Wireless Headphones,Electronics,89.99
1,P002,Bluetooth Speaker,Electronics,49.99
2,P003,Laptop Stand,Electronics,34.99
3,P004,USB-C Hub,Electronics,29.99
4,P005,Smart Watch,Electronics,199.99


In [ ]:
# Improved staging view: trim text, blank string -> NULL, standardise status,
# invalid dates -> NULL, join products for unit_price, derive line_total.
# Grain: one row per order line; order_id repeats for 100 orders (2 lines each).
conn.execute("DROP VIEW IF EXISTS stg_orders;")
conn.execute("""
CREATE VIEW stg_orders AS
SELECT
NULLIF(TRIM(o.order_id), '') AS order_id,
NULLIF(TRIM(o.customer_id), '') AS customer_id,
NULLIF(TRIM(o.product_id), '') AS product_id,
o.quantity AS quantity,
CASE WHEN o.order_date GLOB '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]' THEN o.order_date ELSE NULL END AS order_date,
CASE WHEN TRIM(o.status) = 'IN_TRANSIT!!' THEN 'in_transit' ELSE LOWER(TRIM(o.status)) END AS status,
CAST(p.price AS REAL) AS unit_price,
CAST(o.quantity AS REAL) * CAST(p.price AS REAL) AS line_total
FROM orders_raw o
LEFT JOIN products p ON o.product_id = p.product_id;
""")
conn.commit()
display(pd.read_sql_query('SELECT * FROM stg_orders LIMIT 5;', conn))

,order_id,customer_id,product_id,quantity,order_date,status,unit_price,line_total
0,O02814,C0472,P011,4,2025-07-15,pending,79.99,319.96
1,O04017,C0492,P020,5,2025-04-30,pending,16.99,84.95
2,O03993,C0065,P001,5,2025-01-10,delivered,89.99,449.95
3,O04314,C0107,P011,1,2025-07-03,delivered,79.99,79.99
4,O01732,C0175,P019,2,2024-04-28,shipped,99.99,199.98


### Example transformation pattern — adapt to your schema

This is only an example; it will not run unless the fields exist in your table.

```sql
SELECT
    order_id,
    product_id,
    quantity,
    unit_price,
    CAST(quantity AS REAL) * CAST(unit_price AS REAL) AS line_total
FROM orders_raw;
```

SQLite casts can behave unexpectedly for invalid text values. Inspect the raw values and document conversion rules before relying on calculated values.

## 6. Data-quality checks

The examples below assume `customer_id` exists and that the customers table should have one row per customer. Change the field name or rule to match your actual schema.

In [ ]:
# Data-quality checks on the staging views (Step 6).
# Rule 1: every order line should carry a customer_id.
# Rule 2: customer_id must be unique in stg_customers (one row per customer).
# Rule 3: order_id repeats only where an order genuinely has two lines.
# Rule 4: every order must reference a known customer and product.
# Rule 5: quantity must be positive. Rule 6: order_date must be a real date.
# Rule 7: status labels must be standardised (no IN_TRANSIT!!).
# Interpretation of each result is in the Findings summary at the end.
display(pd.read_sql_query("SELECT COUNT(*) AS missing_customer_id FROM stg_orders WHERE customer_id IS NULL;", conn))
display(pd.read_sql_query("SELECT COUNT(*) AS duplicate_customer_ids FROM (SELECT customer_id FROM stg_customers GROUP BY customer_id HAVING COUNT(*) > 1);", conn))
display(pd.read_sql_query("SELECT COUNT(*) AS orders_with_multiple_rows FROM (SELECT order_id FROM stg_orders GROUP BY order_id HAVING COUNT(*) > 1);", conn))
display(pd.read_sql_query("SELECT COUNT(*) AS orphan_customer_orders FROM stg_orders o LEFT JOIN stg_customers c ON o.customer_id = c.customer_id WHERE o.customer_id IS NOT NULL AND c.customer_id IS NULL;", conn))
display(pd.read_sql_query("SELECT COUNT(*) AS orphan_product_orders FROM stg_orders o LEFT JOIN stg_products p ON o.product_id = p.product_id WHERE p.product_id IS NULL;", conn))
display(pd.read_sql_query("SELECT COUNT(*) AS non_positive_quantity, MIN(quantity) AS min_quantity FROM stg_orders WHERE quantity <= 0;", conn))
display(pd.read_sql_query("SELECT COUNT(*) AS null_order_dates_in_stg FROM stg_orders WHERE order_date IS NULL;", conn))
display(pd.read_sql_query("SELECT COUNT(*) AS invalid_dates_in_raw FROM orders_raw WHERE order_date NOT GLOB '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]';", conn))
display(pd.read_sql_query("SELECT status, COUNT(*) AS n FROM stg_orders GROUP BY status ORDER BY n DESC;", conn))

In [ ]:
# Example: duplicate customer IDs (only if one row per customer is expected).
#
# display(pd.read_sql_query("""
# SELECT customer_id, COUNT(*) AS records
# FROM stg_customers
# GROUP BY customer_id
# HAVING COUNT(*) > 1;
# """, conn))

## 7. Analytical SQL — answer business questions

Use your actual column names. First inspect `stg_orders`, `stg_customers`, and `stg_products` and identify:
- the order identifier and customer identifier
- the product identifier (if each row represents an order line)
- the order date
- the quantity, unit price, discount, and line total (where available)
- any order status, region, or category fields

**Important:** The queries below are templates, not guaranteed to run unchanged. Replace placeholder column names with fields in your files. If a field is not available, explain that limitation and choose a suitable alternative. Run at least four queries and write 1–3 sentences interpreting each result.


In [ ]:
# Query A - overall sales and order-line summary (Step 7).
# line_total comes from stg_orders (quantity x product price).
# Interpretation: 5,100 lines cover 5,000 distinct orders because 100 orders
# have 2 lines each. Average line value is per LINE, not per order, and the
# totals still include the 53 non-positive-quantity lines, so per-order
# averages must aggregate by order_id first.
display(pd.read_sql_query("SELECT COUNT(*) AS order_line_count, COUNT(DISTINCT order_id) AS distinct_orders, SUM(quantity) AS units_sold, ROUND(SUM(line_total), 2) AS total_sales, ROUND(AVG(line_total), 2) AS average_line_value FROM stg_orders;", conn))

In [ ]:
# Query B - product performance (join orders to products).
# Interpretation: ranking by product_sales shows which products drive revenue;
# orders_including_product counts distinct orders, so a product in many small
# orders is not confused with one in a few large orders.
display(pd.read_sql_query("SELECT p.product_name, p.category, SUM(o.quantity) AS units_sold, ROUND(SUM(o.line_total), 2) AS product_sales, COUNT(DISTINCT o.order_id) AS orders_including_product FROM stg_orders AS o JOIN stg_products AS p ON o.product_id = p.product_id GROUP BY p.product_name, p.category ORDER BY product_sales DESC LIMIT 10;", conn))
# Query C - customer-level performance (top 10 by sales).
# LEFT JOIN keeps customers with no orders visible (NULL sales); LIMIT 10 keeps
# the output readable. A high customer_sales with few orders means large baskets.
display(pd.read_sql_query("SELECT c.customer_id, c.first_name, c.last_name, COUNT(DISTINCT o.order_id) AS order_count, ROUND(SUM(o.line_total), 2) AS customer_sales, ROUND(AVG(o.line_total), 2) AS average_line_value FROM stg_customers AS c LEFT JOIN stg_orders AS o ON c.customer_id = o.customer_id GROUP BY c.customer_id, c.first_name, c.last_name ORDER BY customer_sales DESC LIMIT 10;", conn))
# Query D - monthly sales trend (valid dates only).
# Interpretation: months with NULL dates (58 lines) are excluded, so monthly
# totals cover dated lines only; compare months on the same basis.
display(pd.read_sql_query("SELECT substr(order_date, 1, 7) AS order_month, COUNT(DISTINCT order_id) AS order_count, ROUND(SUM(line_total), 2) AS monthly_sales FROM stg_orders WHERE order_date IS NOT NULL GROUP BY substr(order_date, 1, 7) ORDER BY order_month;", conn))
# Query E - order identifiers appearing more than once.
# Interpretation: this is expected line-item grain, not an error - 100 orders
# have 2 lines each. Any order with more than 2 lines would deserve a closer look.
display(pd.read_sql_query("SELECT order_id, COUNT(*) AS rows_per_order FROM stg_orders GROUP BY order_id HAVING COUNT(*) > 1 ORDER BY rows_per_order DESC LIMIT 20;", conn))

## 8. Reconcile your results with `orders_clean.csv`

Treat `orders_clean.csv` as a reference dataset. Do not assume it has the same grain as `orders_raw` or your staging view. First inspect the fields and determine whether each row represents an order or an order line.

Complete these checks:
1. Compare row counts and column names.
2. Check for missing and duplicate business keys in the reference file.
3. Compare totals only when the two datasets use the same grain and the same definition of sales.
4. If both datasets share a reliable key, compare key coverage and identify records present in only one dataset.
5. If you compare numeric values, report the absolute difference and use a sensible rounding tolerance.
6. Explain at least two differences or data-quality issues you found. If the files cannot be compared directly, explain why and define the transformation needed to make them comparable.


In [ ]:
# Reconciliation (Step 8): staging totals vs orders_clean.csv. Business key = order_id.
print('Row counts and columns:')
print('staging lines (stg_orders):', int(pd.read_sql_query('SELECT COUNT(*) AS n FROM stg_orders;', conn).iloc[0]['n']))
print('clean rows:', len(orders_clean_df))
print('clean columns:', list(orders_clean_df.columns))
print('staging columns:', list(pd.read_sql_query('SELECT * FROM stg_orders LIMIT 1;', conn).columns))
print('Missing reference keys:', int(orders_clean_df['order_id'].isna().sum()))
print('Duplicate reference keys:', int(orders_clean_df['order_id'].duplicated().sum()))
print('CLEAN status values:')
print(orders_clean_df['status'].value_counts().to_string())
transformed_df = pd.read_sql_query("""SELECT order_id, ROUND(SUM(line_total), 2) AS calculated_sales, SUM(quantity) AS staging_quantity FROM stg_orders GROUP BY order_id;""", conn)
clean_priced = orders_clean_df.merge(products_df[['product_id', 'price']], on='product_id', how='left')
clean_priced['reference_sales'] = (clean_priced['quantity'] * clean_priced['price']).round(2)
reference_df = clean_priced.groupby('order_id').agg(reference_sales=('reference_sales', 'sum'), reference_quantity=('quantity', 'sum')).reset_index()
reference_df['reference_sales'] = reference_df['reference_sales'].round(2)
comparison = transformed_df.merge(reference_df, on='order_id', how='outer', indicator=True)
print('Key coverage (staging vs clean):')
print(comparison['_merge'].value_counts().to_string())
matched = comparison[comparison['_merge'] == 'both'].copy()
matched['absolute_difference'] = (matched['calculated_sales'] - matched['reference_sales']).abs()
print('Orders matched in both:', len(matched))
print('Orders differing by more than 0.01:', int((matched['absolute_difference'] > 0.01).sum()))
print('Total staging sales (per-order sum):', round(matched['calculated_sales'].sum(), 2))
print('Total clean sales (per-order sum):', round(matched['reference_sales'].sum(), 2))
print('Largest absolute differences (top 10):')
display(matched.sort_values('absolute_difference', ascending=False).head(10))

## Findings summary
- Scope: 500 customers, 20 products, 5,100 raw order lines covering 5,000 distinct orders, order dates from 2024-01 to 2025-12.
- Grain: customers has one row per customer (customer_id, no duplicates); products has one row per product (product_id, no duplicates); orders has one row per order line, so order_id alone is not unique in the raw data: 100 orders have 2 lines each.
- Staging views: stg_customers, stg_products, stg_orders. Transforms applied: text trimmed, blank strings converted to NULL, status labels standardised ('IN_TRANSIT!!' to 'in_transit', lower case), invalid dates converted to NULL, unit_price joined from products, line_total derived as quantity x unit_price.
- Data quality, with interpretation: 173 lines have a missing customer_id, so customer-level analysis must exclude or separately report them; 0 orphan customer or product references, so every non-missing reference resolves; 53 lines have non-positive quantity (minimum -5), which drags sales totals down and should be quarantined or confirmed as returns before reporting; 58 invalid order dates were nulled in staging, so those lines drop out of the monthly trend; no duplicate customer or product keys.
- Analysis: 5,100 lines, 14,947 units, total sales 1,113,264.53, average line value 218.29. Top product: Smart Watch (Electronics), 926 units, 185,190.74. Top customer: C0152 Nikhil Garcia, 13 orders, 6,777.42. Monthly sales range from 37,012.74 (2025-06) to 62,345.85 (2025-08, peak). Because 100 orders have 2 lines, per-order averages must aggregate by order_id first; the average above is per line, not per order.
- Reconciliation: clean has the same 5,000 order_ids as staging (none only in one file) and the same 6 columns; staging adds unit_price and line_total. Clean has 0 missing and 0 duplicate order_ids. At order grain, staging sales are 1,113,264.53 and clean sales are 1,112,132.73, a difference of 1,131.80; 149 of 5,000 orders differ by more than 0.01. Two systematic causes explain the gap: negative raw quantities appear positive in clean (for example O02253: staging -649.95 for quantity -5, clean 649.95 for quantity 5), and orders with two raw lines appear once in clean (for example O03519: staging quantity 10 from two lines, clean quantity 5). Clean status values are only delivered, shipped, pending and cancelled; the 98 raw 'IN_TRANSIT!!' rows have no matching category in clean.
- Caveats: reported totals still include the 53 non-positive-quantity lines; the 58 lines with nulled dates are excluded from the monthly trend; clean is a separately corrected reference file, not a pure aggregation of staging, so small total differences are expected and are explained above rather than indicating missing orders.

## 9. Save your work

Save the notebook to Drive using **File → Save a copy in Drive**. Download the SQLite database before the Colab runtime ends.

In [ ]:
conn.commit()
# Run this when you have finished using the database connection.
# conn.close()

from google.colab import files
files.download('/content/retail_lab.db')

## Student checklist

- [ ] Inspected all four CSV files.
- [ ] Loaded three raw tables into SQLite.
- [ ] Documented actual schemas and table grain.
- [ ] Created and improved all three staging views using actual field names.
- [ ] Ran data-quality checks and explained decisions.
- [ ] Answered at least three business questions with SQL.
- [ ] Compared results with `orders_clean.csv`.
- [ ] Saved the notebook and downloaded `retail_lab.db`.